# Analisa Data Menggunakan Random Forest

## 1. Deskripsi Proyek
Proyek ini bertujuan untuk melakukan analisis data klasifikasi secara menyeluruh menggunakan node-node yang tersedia di **KNIME Analytics Platform**. Dataset yang digunakan adalah **Play Tennis**, yaitu sekumpulan data yang memuat kondisi cuaca seperti *Outlook, Temperature, Humidity*, dan *Wind* untuk memprediksi apakah seseorang dapat bermain tenis (`PlayTennis` sebagai label keberhasilan).

Pada analisis ini dilakukan perbandingan dua algoritma klasifikasi secara visual menggunakan visualisasi struktur KNIME, yaitu:
1. **Decision Tree**
2. **Random Forest**

## 2. Mengenal Decision Tree & Random Forest
- **Decision Tree**: Memecah data ke dalam rumusan kondisi secara berkala (seperti struktur pohon percabangan) untuk membuat putusan tunggal berdasarkan pemilahan fitur yang dominan.
- **Random Forest**: Sesuai dengan namanya (Forest/Hutan), mode algoritma ini merupakan struktur *ansambel* (kumpulan) yang terdiri dari penyebaran banyak model pohon keputusan independen. Setiap spesifik *tree* disuburkan dari sampel yang random.

### Rumus Prediksi Random Forest (Majority Voting)
Setiap *tree* akan menghasilkan prediksi sub-sistem masing-masing, kemudian penarikan kesimpulannya dicetak berdasarkan suara terbanyak atau *majority voting*.

$$
\hat{y} = \text{mode} \{ h_1(x), h_2(x), \dots, h_K(x) \}
$$


Keterangan Rumus:
- $\hat{y}$ = hasil prediksi akhir (Final Prediction)
- $h_i(x)$ = hasil prediksi dari Decision Tree ke-i
- $K$ = jumlah pohon di dalam hutan Random Forest
- $\text{mode}$ = kelas yang paling dominan/banyak muncul

---

## 3. Penjelasan Workflow KNIME dan Detail Setiap Node
Berdasarkan *workflow* dari KNIME, bagan analisis diorganisasikan secara paralel *(branching)* setelah memisahkan sampel. Strategi ini dibuat sedemikian rupa dengan satu alasan dominan: agar model **Decision Tree** dan **Random Forest** sama-sama dicekoki oleh data asuh yang **sangat identik 100%**. Pengujian perbandingan antar kedua model karenanya dianggap mutlak adil.



Mengenai perancangan komponen dan tugas masing-masing node di dalam KNIME:

### 3.1. Excel Reader
Node penjemput ini berguna memanggil material eksternal dan menetapkan eksistansi dataset kita ke dalam lingkup ruang kerja virtual KNIME. Dalam proyek ini, **Excel Reader** dirancang untuk mendeteksi `play_tennis_knime.xlsx`.
- **Sistem Kerja Utama:** Algoritma secara teratur mem-*parsing* baris pertama (yang sengaja ditetapkan menjadi keterangan Header nama kolom), dan memastikan struktur tipe data dibaca akurat sebagai properti string (tulisan target text).

> ![Workflow KNIME](images/ecel.png)

### 3.2. Table Partitioner
Sesudah dijemput, barisan data selanjutnya segera diestrak menjadi dua jenis subset fungsional yang terisolasi melalui modul *Table Partitioner*:
- **Training Data (Data Latih)**: Untuk melatih dan mengenalkan "pengalaman" polanya kepada mesin algoritma.
- **Testing Data (Data Uji)**: Data perawan sebagai bahan tes buta; seukur dari apakah model menangkap inti rumus yang baik pada kondisi dan cuaca yang tidak ikut-ikut dilatih sebelumnya.

**Metode Strategi: Relative Size 80%:** Seperti pada konfigurasinya, sampel disetel untuk ditarik sebesar *80%* data pelatihan, selebihnya sebanyak *20%* dipotong menjadi materi soal tes/uji di penghujung skenario. Dari *14 input PlayTennis*, 11 terbagi ke ranah Training data dan **hanya 3 input** yang masuk ke ranah Testing Data.

>  ![Workflow KNIME](images/part.png)

### 3.3. Decision Tree Learner
Data pelatikan dari *Partitioner* dirambatkan ke blok spesialis **Decision Tree Learner**.
- Algoritma di dalam node ini mengambil porsi teratas struktur hirarki pada properti atribut subjek, misalnya: *Apakah cerah/hujan?*, lalu bercabang dan memisahkan kondisinya demi memaksilakan kriteria seperti nilai algoritma *Information Gain* (mereduksi error kebingungan prediksi `Yes/No`).


### 3.4. Decision Tree Predictor
Usai otak/model pohon pengetahuannya jadi, model dirantai ke **Predictor**. 
- Predictor tidak lagi memiliki kuasa mengubah struktur pelajaran algoritma. Sebaliknya dia menelan pasokan data buta *3 baris Testing data* mentah kemudian meng-*append* atau menempelkan satu kolom paling kanan. Kolom tersebut berteriak tentang terkaan mesin (apakah ia melabeli `Yes` atau tidak berdasarkan patokan pohon di *Learner*).

> ![Workflow KNIME](images/treee.png)

### 3.5. Random Forest Learner
Ini node pemicu persaingannya. Menjadi rivalitas yang kokoh untuk penarikan satu pohon semata. Algoritma Random Forest Learner menggandakan pohon-pohon tiruan ke dalam jumlah ekstrem.
- Lewat penedekatan mekanisme *Bootstrapping*, ia mengambil secacah porsi random dataset (*dengan pengembalian*) untuk menyusun lebih dari puluhan bahkan ratusan pohon cuaca perorangan di titik node (*Ensemble*).
- **Manfaatnya:** Random Forest menjadi sulit termanipulasi kebisingan (*Noise Data*) karena jika suatu pohon salah tanggap, ratusan teman pohon lainnya membawakan cadangan penyeimbangnya, menjadikan mode ini nyaris kebal akan anomali pola (*Overfitting*).

> ![Workflow KNIME](images/forest.png)

### 3.6. Random Forest Predictor
Sama-sama difungsikan untuk menebak properti tes acak, ia dioperasikan dalam model *forest*. Menggunakan konsep pemusyawarahan (*Majority Voting*).
- Cara Kerja: Setiap kali 1 titik soal cuaca diajukan, seluruh populasi "hutan" mesin melakukan voting kelas. Jika 85 dari 100 anggotanya menyuarakan *'Yes'* sementara 15 lainnya sepakat memprediksi *'No'*, otomatis label hasil prediksi *PlayTennis* dimantapkan di kesimpulan `Yes` yang masif.

> ![Workflow KNIME](images/forest.png)

### 3.7. Scorer
Scorer adalah node wasit dan hakim utama di dalam siklus analitiks. Fungsinya bukan lagi menebak, tetapi menjajarkan kolom aktual *(Ground Truth asli dari dataset)* dan memadukannya dengan kolom *(Predicted)* dari Predictor secara komparatif rasional. Hasilnya dituangkan dalam bentuk *Confusion Matrix* yang menjadi pandasi evaluasi di bawah sana.

> ![Workflow KNIME](images/acc.png)

---

## 4. Evaluasi Hasil Scorer (Berdasarkan Hasil Analisis KNIME)
Menganalisa matriks keakuratan *(Confusion matrix)* pada hasil temuan output evaluator (Scorer), khususnya saat membedah blok Random Forest Predictor untuk perbandingan, bisa diamati kalkulasinya seperti berikut:

Konteks *Testing Data = 3 Catatan (Baris)*
- Total ditebak persis (*Prediksi Yes; Nyata Yes / True Positive*) = **2 data** 
- Total diyakini salah (*Prediksi No; Nyata Yes / False Negative*) = **1 data**

**Tingkat Kalkulasi Metrik Dominannya:**
- **Accuracy (Akurasi)**: 
Seberapa banyak mesin menebak pas? Di titik ini nilainya adalah total ketepatan berbanding lurus dengan seluruh bahan ujinya.

$$ 
\text{Accuracy} = \frac{2}{3} = 0.666 \; (66.67\%) 
$$


- **Error Rate (Angka Risiko Ketidakmampuan Model)**:
Persentase di mana mesin gagal paham menebak secara luput.

$$ 
\text{Error Rate} = \frac{1}{3} = 0.333 \; (33.33\%) 
$$


## 5. Kesimpulan Proyek Analisis
Pada rekam hasil visual dari performa rasio Scorer tersebut, margin Akurasi sebesar **66.67%** merupakan konklusi yang logis secara alamiah ketika menyandingkan data kecil. Dari *14 total data mentah*, alokasi sebesar *20% pengujian acak* menelurkan porsi hanya sebatas **3 Record uji**.

Karena jumlah data uji terlampau dangkal, kehilangan kemampuan identifikasi *satu jenis label cuaca target saja*, memprakarsai kerusakan drastis (*tumbangnya 33.33% kredibilitas keakuratan evaluator*). Meskipun demikian, pola pembagian perbandingan (Node terpisah) ini merupakan *Standard of Practice* (SOP) terbaik KNIME dalam mengobservasi dan berburu model berkinerja tertinggi. Random Forest umumnya lebih tangguh dipakai untuk mencegah halusinasi pola pada data *big scale*, namun Decision Tree senantiasa kompeten meretas data *dataset* semacam Play Tennis yang tak begitu pelik kondisinya.